# Phase 3 — FID-50K Validation

Replaces the noisy FID-10K estimates with paper-grade FID-50K numbers for the headline configs that go in the May 3 progress report.

## Configs (5 runs total)

| # | Config | Steps | Why |
|---|--------|-------|-----|
| 1 | vanilla | 16 | matched-step baseline (priority) |
| 2 | margin τ=0.5 cap=0.5 | 16 | **headline winner** (priority) |
| 3 | vanilla | 32 | enables "approaches 32-step quality" claim (bonus) |
| 4 | vanilla | 8 | aggressive baseline (bonus) |
| 5 | margin τ=0.5 cap=0.5 | 8 | validates the 17.6% claim at FID-50K (bonus) |

**Runtime: ~80–100 min** sampling on A100 + ~25 min FID eval.

## I/O strategy (important)

- **PNGs** (50,000 small files per config) → **local disk** (`/content/phase3-local/`) for fast I/O.
- **Final NPZs** → **Drive** (`/content/drive/MyDrive/ARPG-assets/results/pilot-20260421/phase3-fid50k/`) for persistence.
- After each config completes, the script syncs the single ~10 GB NPZ to Drive (~5 min) and deletes the local PNG folder to free disk.

This avoids the 30× slowdown that comes from writing 50,000 small PNG files directly to Drive (Drive is slow for many-small-file writes).

Resumable — re-running skips configs whose NPZ is already in Drive.

## 1. Setup

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    raise RuntimeError("No GPU! Runtime > Change runtime type > A100 GPU")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ARPG = "/content/drive/MyDrive/ARPG-assets"

import os
PHASE3_DIR = f"{DRIVE_ARPG}/results/pilot-20260421/phase3-fid50k"
os.makedirs(PHASE3_DIR, exist_ok=True)

# Confirm previous combined results exists for later merging
PREV_COMBINED = f"{DRIVE_ARPG}/results/pilot-20260421/phase2-16step/combined-results-v2.csv"
print(f"Phase 3 output dir: {PHASE3_DIR}")
print(f"Previous combined CSV exists: {os.path.exists(PREV_COMBINED)}")

In [ ]:
os.chdir('/content')
!rm -rf /content/ARPG
!git clone https://github.com/rshahbazov23/comp447-arpg-private.git /content/ARPG
%cd /content/ARPG

# Symlink heavy assets from Drive
!ln -sfn {DRIVE_ARPG}/weights weights
!ln -sfn {DRIVE_ARPG}/eval eval
!ln -sfn {DRIVE_ARPG}/external external

!pip install -q transformers einops Pillow tqdm numpy scipy tensorflow pandas

In [ ]:
# Sanity check
required = [
    "weights/arpg_300m.pt",
    "weights/vq_ds16_c2i.pt",
    "eval/VIRTUAL_imagenet256_labeled.npz",
    "external/guided-diffusion/evaluations/evaluator.py",
    "scripts/run_phase3_validation.sh",
    "scripts/eval_pilot_sweep.py",
]
for p in required:
    status = "OK" if os.path.exists(p) else "MISSING"
    print(f"  [{status}] {p}")

## 2. Run the FID-50K sweep

The script samples to local disk (fast), then syncs the final NPZ to Drive after each config completes.

**Time estimate per config (50K samples, batch=64 on A100):**
- 8 steps:   ~3–5 min sampling + ~3–5 min Drive sync
- 16 steps: ~10–12 min sampling + ~3–5 min Drive sync
- 32 steps: ~20–22 min sampling + ~3–5 min Drive sync

**Total ~80–100 min for all 5 configs.**

Set `RUN_BONUS=0` to skip configs 3–5 (priority-only mode).

In [ ]:
env = {
    'PHASE3_DIR':           PHASE3_DIR,
    'NUM_SAMPLES':          '50000',
    'PER_PROC_BATCH_SIZE':  '64',
    'CFG_SCALE':            '5.0',
    'GLOBAL_SEED':          '0',
    'RUN_BONUS':            '1',   # set to '0' to run only the 2 priority configs
}
env_prefix = ' '.join(f"{k}='{v}'" for k, v in env.items())

!{env_prefix} bash scripts/run_phase3_validation.sh

In [ ]:
# Verify NPZ counts
import glob
vanilla_npzs = sorted(glob.glob(f"{PHASE3_DIR}/vanilla/*.npz"))
rejection_npzs = sorted(glob.glob(f"{PHASE3_DIR}/rejection/*.npz"))
print(f"Vanilla NPZs:   {len(vanilla_npzs)}")
print(f"Rejection NPZs: {len(rejection_npzs)}")
for f in vanilla_npzs + rejection_npzs:
    print(f"  {os.path.basename(f)}")

## 3. FID-50K evaluation

Reading 10 GB NPZs from Drive is slower than from local disk, but each FID eval still completes in under 6 minutes. Total: ~25 min for 5 configs.

In [ ]:
PHASE3_CSV = f"{PHASE3_DIR}/fid50k-results.csv"

!python scripts/eval_pilot_sweep.py \
    --pilot-dir {PHASE3_DIR} \
    --reference-npz eval/VIRTUAL_imagenet256_labeled.npz \
    --guided-diffusion external/guided-diffusion/evaluations/evaluator.py \
    --out-csv {PHASE3_CSV}

## 4. Headline results table

Build the table that goes in the progress report.

In [ ]:
import pandas as pd
import re

def extract_steps(npz_name):
    m = re.search(r'step-(\d+)', npz_name)
    return int(m.group(1)) if m else None

df = pd.read_csv(PHASE3_CSV)
df['steps'] = df['npz'].apply(extract_steps)
df_ok = df[df['fid'].notna()].copy().sort_values(['steps', 'mode'])

print("=== Phase 3 FID-50K results ===\n")
cols = ['steps', 'mode', 'metric', 'tau', 'cap', 'fid', 'is_score', 'precision', 'recall']
df_ok[cols].round(3)

In [ ]:
# Build the headline comparison table for the progress report
vanilla_rows = df_ok[df_ok['mode'] == 'vanilla'].set_index('steps')
rejection_rows = df_ok[df_ok['mode'] == 'rejection'].set_index('steps')

print("=" * 70)
print("PROGRESS REPORT HEADLINE TABLE (FID-50K, ARPG-L on ImageNet-256)")
print("=" * 70)
print(f"{'steps':>6} {'vanilla':>10} {'rejection':>12} {'delta':>10} {'pct':>8}")
print('-' * 50)
common_steps = sorted(set(vanilla_rows.index) & set(rejection_rows.index))
for s in common_steps:
    v = vanilla_rows.loc[s, 'fid']
    r = rejection_rows.loc[s, 'fid']
    delta = r - v
    pct = (r - v) / v * 100
    marker = "  <--" if delta < 0 else ""
    print(f"{s:>6} {v:>10.3f} {r:>12.3f} {delta:>+10.3f} {pct:>+7.2f}%{marker}")

# Vanilla-only at 32 steps (no rejection counterpart at 32 steps in Phase 3)
print()
for s in sorted(set(vanilla_rows.index) - set(rejection_rows.index)):
    v = vanilla_rows.loc[s, 'fid']
    print(f"{s:>6} {v:>10.3f} {'(none)':>12}")

## 5. FID-10K vs FID-50K: do the headline numbers hold up?

In [ ]:
# Pull the FID-10K numbers we have for the same configs from previous phases
if os.path.exists(PREV_COMBINED):
    prev = pd.read_csv(PREV_COMBINED)
    if 'steps' not in prev.columns:
        prev['steps'] = prev['npz'].apply(extract_steps)

    print("=== FID-10K vs FID-50K comparison ===\n")
    print(f"{'config':<55} {'FID-10K':>10} {'FID-50K':>10} {'shift':>8}")
    print('-' * 90)

    for _, p in df_ok.iterrows():
        s = int(p['steps'])
        if p['mode'] == 'vanilla':
            label = f"vanilla / step={s}"
            match = prev[(prev['mode']=='vanilla') & (prev['steps']==s)]
        else:
            label = f"rejection / step={s} {p['metric']} tau={p['tau']} cap={p['cap']}"
            match = prev[(prev['mode']=='rejection') &
                         (prev['steps']==s) &
                         (prev['metric']==p['metric']) &
                         (prev['tau']==p['tau']) &
                         (prev['cap']==p['cap'])]
        fid10k = match['fid'].iloc[0] if len(match) else None
        fid50k = p['fid']
        if fid10k is not None:
            shift = fid50k - fid10k
            print(f"{label:<55} {fid10k:>10.3f} {fid50k:>10.3f} {shift:>+8.3f}")
        else:
            print(f"{label:<55} {'(no FID-10K)':>10} {fid50k:>10.3f}")
else:
    print("(Previous combined CSV not found — skipping FID-10K comparison.)")

## 6. Compute-efficiency claim

If we have vanilla at 16/32 steps AND rejection at 16 steps, we can express the result as: "rejection at 16 steps closes X% of the gap between vanilla 16 and vanilla 32."

In [ ]:
v16 = vanilla_rows.loc[16, 'fid'] if 16 in vanilla_rows.index else None
v32 = vanilla_rows.loc[32, 'fid'] if 32 in vanilla_rows.index else None
r16 = rejection_rows.loc[16, 'fid'] if 16 in rejection_rows.index else None

if all(x is not None for x in [v16, v32, r16]):
    gap = v16 - v32              # how much worse 16-step vanilla is than 32-step vanilla
    closed = v16 - r16            # how much our method improves over 16-step vanilla
    print("=" * 60)
    print("COMPUTE-EFFICIENCY CLAIM")
    print("=" * 60)
    print(f"vanilla, 16 steps:    FID = {v16:.3f}")
    print(f"vanilla, 32 steps:    FID = {v32:.3f}")
    print(f"rejection, 16 steps:  FID = {r16:.3f}")
    print()
    print(f"Gap (16-step → 32-step quality): {gap:.3f}")
    print(f"Closed by rejection at 16 steps: {closed:.3f}")
    print(f"Fraction of gap closed:           {closed / gap * 100:.1f}%")
else:
    print("Need vanilla 16, vanilla 32, and rejection 16 to compute this.")
    print(f"Have: v16={v16}, v32={v32}, r16={r16}")

## 7. Save everything to Drive (final consolidated CSV)

In [ ]:
# Combine all phases into a single master CSV
frames = []
if os.path.exists(PREV_COMBINED):
    p = pd.read_csv(PREV_COMBINED)
    if 'steps' not in p.columns:
        p['steps'] = p['npz'].apply(extract_steps)
    if 'eval_type' not in p.columns:
        p['eval_type'] = 'FID-10K'
    frames.append(p)

df_phase3 = df_ok.copy()
df_phase3['eval_type'] = 'FID-50K'
df_phase3['source'] = 'phase3'
frames.append(df_phase3)

master = pd.concat(frames, ignore_index=True)
master = master.drop_duplicates(subset=['npz'], keep='last').reset_index(drop=True)
MASTER_CSV = f"{PHASE3_DIR}/master-results.csv"
master.to_csv(MASTER_CSV, index=False)
print(f"Master CSV with all phases: {MASTER_CSV}")
print(f"Total rows: {len(master)}")
print(f"  FID-10K rows: {len(master[master['eval_type']=='FID-10K'])}")
print(f"  FID-50K rows: {len(master[master['eval_type']=='FID-50K'])}")

## 8. Summary for the progress report

Print a paste-ready summary block.

In [ ]:
print("=" * 70)
print("PROGRESS REPORT - PASTE-READY SUMMARY")
print("=" * 70)
print()
print("Vanilla baselines (FID-50K):")
for s in sorted(vanilla_rows.index):
    v = vanilla_rows.loc[s, 'fid']
    print(f"  ARPG-L vanilla / {s} steps: FID = {v:.3f}")
print()
print("Rejection results (FID-50K, margin / tau=0.5 / cap=0.5):")
for s in sorted(rejection_rows.index):
    r = rejection_rows.loc[s, 'fid']
    v = vanilla_rows.loc[s, 'fid'] if s in vanilla_rows.index else None
    if v is not None:
        delta = r - v
        pct = delta / v * 100
        print(f"  ARPG-L rejection / {s} steps: FID = {r:.3f}  (Δ = {delta:+.3f}, {pct:+.2f}%)")
    else:
        print(f"  ARPG-L rejection / {s} steps: FID = {r:.3f}")
print()
print(f"Existing (Phase 0): vanilla / 64 steps FID-50K = 2.42 (matches paper 2.37)")
print()
print("=" * 70)